# Colab pipeline for SciML_CFD_Engine
This notebook sets up the environment, launches TensorBoard, lets you override key hyperparameters, runs training, and visualizes a pipe cross-section from the latest checkpoint.

Replace `YOUR_REPO_URL` below with your GitHub repository URL before running the first cell.

In [ ]:
# 1) Clone repository (replace YOUR_REPO_URL) and install requirements
!git clone YOUR_REPO_URL repo || true
%cd repo || true
# Install standard requirements; in Colab you may want the CUDA-enabled wheel for torch
!pip install -r cfd_engine/requirements.txt -q || true
# If torch not installed or GPU desired, uncomment and run the appropriate install command below.
# For Colab GPU (example - adjust CUDA version if needed):
# !pip install --upgrade pip
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# 2) Start TensorBoard integration for Colab
%load_ext tensorboard
import os
os.environ.setdefault('LOGDIR', '/content/logs')
%tensorboard --logdir $LOGDIR --host 0.0.0.0 --port 6006

# 3) Hyperparameter dashboard and training run
Use the cell below to set runtime parameters. These values are passed to `main.py` via environment variables. Edit values as needed and then run the cell to begin training.

In [ ]:
# Set hyperparameters (edit before running)
import os
# Logging directory for TensorBoard
os.environ['LOGDIR'] = '/content/logs'
# Physics / training parameters
os.environ['RE'] = '1000'
os.environ['ADAM_EPOCHS'] = '200'
os.environ['LBFGS_EPOCHS'] = '50'
os.environ['BATCH_INTERIOR'] = '2000'
os.environ['BATCH_BOUNDARY'] = '400'
os.environ['NTK_REG_WEIGHT'] = '1e-4'
os.environ['NTK_CHECK_INTERVAL'] = '1000'
os.environ['LAMBDA_POS'] = '10.0'
os.environ['LAMBDA_PIN'] = '1.0'
os.environ['LAMBDA_BC'] = '20.0'
os.environ['INLET_VELOCITY'] = '1.0'
os.environ['RADIUS'] = '0.5'
os.environ['LENGTH'] = '3.0'

# Run training (this will stream output in the notebook). Depending on your runtime, this may take a long time.
# If you prefer to run in background, use nohup or screen equivalents.
!python cfd_engine/main.py

# 4) Post-processing: load latest checkpoint and visualize a pipe cross-section
After training completes (or when a checkpoint exists), run the cell below to load the most recent checkpoint and plot `u` over a circular cross-section at a given `x` slice.

In [ ]:
import os, glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.networks import PINN3DEngine
from src.geometry.sdf_sampler import PipeGeometrySampler

# Find latest checkpoint
ckpt_dir = 'checkpoints'
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, '*.pth')))

,